# 🌙 Operation Night Watch: The Village Outpost — *Mock Task M1*

> **Format-faithful practice task** in the style of the IOAI 2026 At-Home Round (Home Task 1 family).
> Built for contest-condition practice: solve it using **only the Gemma chatbot** for help, then send
> `submission.csv` + this notebook back for grading against a hidden test set.

## The story

The village of Karatau runs a cheap acoustic sensor network. Its tiny on-device classifier was trained to
recognise **8 everyday sounds** (rain, wind, dog barks, car horns, footsteps, door knocks, birdsong, idling engines).

The rangers now need it to also detect **5 new sounds**: *chainsaw* (illegal logging), *gunshot* (poaching),
*cricket* and *mosquito* (insect monitoring), and *glass_break* (outpost intrusion).

You are given:
- `base_model.pt` — the deployed model, trained on the **old 8 classes only** (you do **not** get its full training data),
- `data/train_old.npz` — a **small retained subset** of old-class clips (25 per class),
- `data/fine_tune.npz` — the new-class clips (**imbalanced**: 12–40 per class),
- `data/val.npz` — a labeled validation set over all 13 classes,
- `data/test.npz` — the unlabeled test clips you must predict.

Each "clip" is a 32-band × 64-frame log-mel-style spectrogram (`float32`, shape `(32, 64)`).

## ⚖️ Scoring (read this before anything else)

```
Score = 0.5 · Accuracy(old 8 classes) + 0.5 · Accuracy(new 5 classes)
```

computed on a **hidden test set**. A model that nails the new sounds but forgets the old ones scores ~nothing.
The concept this task teaches: **catastrophic forgetting**.

## 1 · Setup & data

Runs on CPU in a couple of minutes — no GPU needed. If the `data/` folder isn't next to this notebook
(e.g. you're on Colab), the cell clones the repo and uses its copy.

In [ ]:
import json, os, pathlib
import numpy as np
import torch, torch.nn as nn

TASK_DIR = pathlib.Path(".")
if not (TASK_DIR / "data" / "classes.json").exists():
    if not pathlib.Path("ioai-prep").exists():
        !git clone -q https://github.com/tapiwamakandigona/ioai-prep
    TASK_DIR = pathlib.Path("ioai-prep/mock/M1_village_outpost")

classes = json.load(open(TASK_DIR / "data" / "classes.json"))
OLD, NEW = classes["old_classes"], classes["new_classes"]
ALL = classes["all_classes"]
N_OLD, N_ALL = len(OLD), len(ALL)

train_old = np.load(TASK_DIR / "data" / "train_old.npz")
fine_tune = np.load(TASK_DIR / "data" / "fine_tune.npz")
val       = np.load(TASK_DIR / "data" / "val.npz")
test_X    = np.load(TASK_DIR / "data" / "test.npz")["X"]

print("old classes:", OLD)
print("new classes:", NEW)
print("train_old:", train_old["X"].shape, "| fine_tune:", fine_tune["X"].shape,
      "| val:", val["X"].shape, "| test:", test_X.shape)

## 2 · Look at the data

Class balance and what the "spectrograms" look like. **Actually look** — imbalance and class
signatures are where points hide.

In [ ]:
import matplotlib.pyplot as plt

vals, counts = np.unique(fine_tune["y"], return_counts=True)
print("fine-tune clips per NEW class:")
for v, c in zip(vals, counts):
    print(f"  {ALL[v]:<12} {c}")

fig, axes = plt.subplots(2, 4, figsize=(14, 5))
show = ["rain", "dog_bark", "birdsong", "engine_idle", "chainsaw", "gunshot", "cricket", "glass_break"]
for ax, cls in zip(axes.flat, show):
    pool = val if ALL.index(cls) >= N_OLD or True else train_old
    i = np.where(val["y"] == ALL.index(cls))[0][0]
    ax.imshow(val["X"][i], aspect="auto", origin="lower", cmap="magma")
    ax.set_title(cls, fontsize=9); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## 3 · The provided model & baseline

`TinySpecCNN` mirrors the deployed sensor model: a small conv encoder + a linear classification head.
`base_model.pt` holds weights trained on the **old 8 classes only**.

**The baseline below is intentionally naive.** It builds a fresh 13-class model, loads only the
pretrained **encoder** weights, and fine-tunes the whole thing on the **new-class clips only**.
Its known limitations (a.k.a. your improvement checklist) are listed in section 5.

In [ ]:
class TinySpecCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(16 * 8 * 16, 64), nn.ReLU(),
        )
        self.head = nn.Linear(64, n_classes)
    def forward(self, x):
        return self.head(self.encoder(x))


def train_model(model, X, y, epochs=12, lr=1e-3, batch=32, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss()
    ds = torch.utils.data.TensorDataset(torch.tensor(X).unsqueeze(1), torch.tensor(y))
    dl = torch.utils.data.DataLoader(ds, batch_size=batch, shuffle=True,
                                     generator=torch.Generator().manual_seed(seed))
    model.train()
    for ep in range(epochs):
        tot = 0.0
        for xb, yb in dl:
            opt.zero_grad()
            loss = lossf(model(xb), yb)
            loss.backward()
            opt.step()
            tot += loss.item() * len(yb)
        if ep % 4 == 0 or ep == epochs - 1:
            print(f"epoch {ep}: loss {tot / len(ds):.4f}")
    return model


@torch.no_grad()
def predict(model, X, batch=64):
    model.eval()
    out = []
    for i in range(0, len(X), batch):
        out.append(model(torch.tensor(X[i:i + batch]).unsqueeze(1)).argmax(1))
    return torch.cat(out).numpy()

In [ ]:
# ---- BASELINE (naive — improve me!) ----
base_state = torch.load(TASK_DIR / "base_model.pt", map_location="cpu")

torch.manual_seed(0)
model = TinySpecCNN(N_ALL)
enc_state = {k: v for k, v in base_state.items() if k.startswith("encoder.")}
model.load_state_dict(enc_state, strict=False)   # old HEAD weights are thrown away!

model = train_model(model, fine_tune["X"], fine_tune["y"], epochs=12)

## 4 · Metric

The exact scoring function used on the hidden test set (there, over 40 clips per class).

In [ ]:
def competition_score(y_true, y_pred, n_old=N_OLD):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    old, new = y_true < n_old, y_true >= n_old
    acc_old = (y_pred[old] == y_true[old]).mean()
    acc_new = (y_pred[new] == y_true[new]).mean()
    return 0.5 * acc_old + 0.5 * acc_new, acc_old, acc_new

val_pred = predict(model, val["X"])
score, acc_old, acc_new = competition_score(val["y"], val_pred)
print(f"VALIDATION  score={score:.3f}   acc_old={acc_old:.3f}   acc_new={acc_new:.3f}")

print("\nper-class validation accuracy:")
for k, cls in enumerate(ALL):
    m = val["y"] == k
    tag = "old" if k < N_OLD else "NEW"
    print(f"  [{tag}] {cls:<12} {(val_pred[m] == k).mean():.2f}")

## 5 · Your mission

Beat the baseline. **Baseline validation score ≈ 0.43** (acc_old = 0.00 — total forgetting) — write your
number down after running it. Target: **≥ 0.85 on validation**; a good solution lands **≈ 0.80+ on the
hidden test** (expect val→test drop of a few points). The baseline's stated weaknesses are the intended
solution paths:

1. **It throws away the old head.** Expand the head 8 → 13 instead: create the 13-class head and **copy the
   old head's weights/bias into the first 8 rows** before fine-tuning.
2. **It never shows the model an old clip.** You *have* `train_old.npz` — mix old clips into the fine-tuning
   batches (**experience replay**). The old:new ratio is a knob worth tuning.
3. **It fine-tunes everything at full LR.** Consider freezing the encoder (or giving it a much smaller
   learning rate than the head).
4. The fine-tune set is **imbalanced** (12–40 clips/class) — class weighting or oversampling may help the rare ones.
5. Diagnose before you fix: the per-class table above tells you *where* the points are lost.

### Rules of the game (contest conditions)
- Help allowed: **the Gemma chatbot only** (no web search, no other AI, docs of installed libs are fine).
- One change at a time; keep a scrappy log of `change → validation score`.
- Suggested time box: **90 minutes** from opening this notebook to final `submission.csv`.
- Don't touch section 4's metric cell; never hand-edit `submission.csv` values.

## 6 · Write the submission

Predict the hidden test set and write `submission.csv` (`clip_id,label` with class **names**).
Run this early — submit the plain baseline first, points on the board + format verified.

In [ ]:
test_pred = predict(model, test_X)
import csv
with open("submission.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["clip_id", "label"])
    for i, p in enumerate(test_pred):
        w.writerow([i, ALL[p]])
print(f"wrote submission.csv ({len(test_pred)} rows)")
print(open("submission.csv").read(120))

## 7 · Constraints & submitting

- **Hidden test set**: 40 clips × 13 classes; graded with the exact `competition_score` above.
- **Compute**: everything here runs on CPU in minutes — if a training run takes > ~5 min, you're overdoing it.
- **Submission**: send the updated notebook + `submission.csv` back to Tapiwa in Slack. Grading returns
  `score`, `acc_old`, `acc_new` (and the baseline's numbers for comparison). Best of 3 submissions counts.
- Validation ≈ hidden test in distribution, but **expect a small gap** — don't over-tune to 130 val clips.